# Train Number Plate YOLO Detector in Google Colab

This notebook is separate from the Flask traffic violation project. It trains a YOLO detection model for **license plates only** and exports the trained weights as `plate_best.pt`.

Do not replace your existing violation model `best.pt` with this file. The output from this notebook is intended to be integrated later as a second model for plate detection.

## 1. Mount Google Drive

The final trained model will be copied to Google Drive so it is not lost when the Colab runtime resets.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Install Ultralytics YOLO

This installs Ultralytics and Roboflow support. Use a GPU runtime for training: `Runtime > Change runtime type > T4 GPU` or better.

In [ ]:
!pip install -q ultralytics roboflow

import ultralytics
ultralytics.checks()

## 3. Required Dataset Format

Use a YOLO detection dataset with one class for plates, for example `license_plate` or `number_plate`.

Expected structure:

```text
dataset/
  data.yaml
  train/
    images/
    labels/
  valid/        # or val/
    images/
    labels/
  test/         # optional
    images/
    labels/
```

`data.yaml` should point to the image folders and include the class names, for example:

```yaml
train: train/images
val: valid/images
test: test/images
nc: 1
names: ['license_plate']
```

## 4. Configure Paths

If your dataset extracts somewhere else, change `DATASET_DIR` and `DATA_YAML` below.

In [ ]:
from pathlib import Path

DATASET_DIR = Path('/content/dataset')
DATA_YAML = DATASET_DIR / 'data.yaml'

DRIVE_OUT_DIR = Path('/content/drive/MyDrive/plate_yolo_training')
DRIVE_OUT_DIR.mkdir(parents=True, exist_ok=True)

print('Dataset directory:', DATASET_DIR)
print('data.yaml path:', DATA_YAML)
print('Drive output directory:', DRIVE_OUT_DIR)

## 5A. Upload a YOLO Dataset ZIP

Use this section if you exported a YOLO dataset ZIP manually. Upload the ZIP when prompted. The notebook will extract it into `/content/dataset`.

If your ZIP contains a nested folder, update `DATASET_DIR` and `DATA_YAML` after extraction.

In [ ]:
from google.colab import files
import shutil
import zipfile

uploaded = files.upload()

for filename in uploaded.keys():
    if filename.lower().endswith('.zip'):
        if DATASET_DIR.exists():
            shutil.rmtree(DATASET_DIR)
        DATASET_DIR.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(filename, 'r') as zip_ref:
            zip_ref.extractall(DATASET_DIR)
        print('Extracted:', filename, 'to', DATASET_DIR)

print('Top-level files/folders:')
for path in sorted(DATASET_DIR.glob('*')):
    print('-', path)

## 5B. Optional: Import Dataset from Roboflow

Use this section instead of ZIP upload if your dataset is on Roboflow.

Steps:
1. Set `USE_ROBOFLOW = True`.
2. Paste your Roboflow API key.
3. Set workspace, project, and version.
4. Run the cell.

Roboflow's YOLOv8 export format works with Ultralytics YOLO training, including YOLO11 models.

In [ ]:
USE_ROBOFLOW = False

if USE_ROBOFLOW:
    from roboflow import Roboflow

    ROBOFLOW_API_KEY = 'PASTE_YOUR_API_KEY_HERE'
    ROBOFLOW_WORKSPACE = 'your-workspace'
    ROBOFLOW_PROJECT = 'your-project'
    ROBOFLOW_VERSION = 1

    rf = Roboflow(api_key=ROBOFLOW_API_KEY)
    project = rf.workspace(ROBOFLOW_WORKSPACE).project(ROBOFLOW_PROJECT)
    dataset = project.version(ROBOFLOW_VERSION).download('yolov8', location=str(DATASET_DIR))

    DATASET_DIR = Path(dataset.location)
    DATA_YAML = DATASET_DIR / 'data.yaml'

    print('Roboflow dataset location:', DATASET_DIR)
    print('data.yaml path:', DATA_YAML)

## 6. Verify Dataset Structure

Run this before training. If it fails, fix the dataset path or `data.yaml` path first.

In [ ]:
def count_files(folder, patterns):
    folder = Path(folder)
    if not folder.exists():
        return 0
    total = 0
    for pattern in patterns:
        total += len(list(folder.glob(pattern)))
    return total

print('Dataset directory exists:', DATASET_DIR.exists())
print('data.yaml exists:', DATA_YAML.exists())

for split in ['train', 'valid', 'val', 'test']:
    images_dir = DATASET_DIR / split / 'images'
    labels_dir = DATASET_DIR / split / 'labels'
    image_count = count_files(images_dir, ['*.jpg', '*.jpeg', '*.png', '*.bmp', '*.webp'])
    label_count = count_files(labels_dir, ['*.txt'])
    print(f'{split:>5}: images={image_count}, labels={label_count}')

assert DATA_YAML.exists(), f'Missing data.yaml at {DATA_YAML}'

## 7. Check `data.yaml`

This prints the dataset config and verifies that class names and train/validation paths exist.

In [ ]:
import yaml

with open(DATA_YAML, 'r') as f:
    data_config = yaml.safe_load(f)

print(data_config)

assert 'names' in data_config, 'data.yaml must include names'
assert 'train' in data_config, 'data.yaml must include train path'
assert 'val' in data_config or 'valid' in data_config, 'data.yaml should include val path'

names = data_config['names']
if isinstance(names, dict):
    names = list(names.values())

print('Classes:', names)
print('Class count:', len(names))

if len(names) != 1:
    print('Warning: For plate-only detection, the dataset should usually have exactly one class.')

## 8. Train YOLO Plate Detector

This trains a small YOLO11 model. Increase `epochs` if validation metrics are still improving. Keep `imgsz=640` unless your plates are very small and you have enough GPU memory.

In [ ]:
EPOCHS = 50
IMG_SIZE = 640
BASE_MODEL = 'yolo11n.pt'
PROJECT_DIR = '/content/runs/plate_detector'
RUN_NAME = 'license_plate_yolo11n'

!yolo task=detect mode=train model={BASE_MODEL} data={DATA_YAML} epochs={EPOCHS} imgsz={IMG_SIZE} plots=True project={PROJECT_DIR} name={RUN_NAME}

## 9. Locate Trained Weights

Ultralytics saves the best checkpoint as `weights/best.pt` inside the run directory.

In [ ]:
BEST_PT = Path(PROJECT_DIR) / RUN_NAME / 'weights' / 'best.pt'
LAST_PT = Path(PROJECT_DIR) / RUN_NAME / 'weights' / 'last.pt'

print('best.pt:', BEST_PT, 'exists=', BEST_PT.exists())
print('last.pt:', LAST_PT, 'exists=', LAST_PT.exists())

assert BEST_PT.exists(), 'Training did not produce best.pt. Check the training logs above.'

## 10. Validate Model

Validation reports precision, recall, and mAP. Do not treat the model as ready until these numbers are acceptable on your own validation data.

In [ ]:
!yolo task=detect mode=val model={BEST_PT} data={DATA_YAML} imgsz={IMG_SIZE}

## 11. Run Prediction on Sample Images

This runs detection on test images if available, otherwise validation images, otherwise training images.

In [ ]:
candidate_sources = [
    DATASET_DIR / 'test' / 'images',
    DATASET_DIR / 'valid' / 'images',
    DATASET_DIR / 'val' / 'images',
    DATASET_DIR / 'train' / 'images',
]

SAMPLE_SOURCE = None
for source in candidate_sources:
    if source.exists() and count_files(source, ['*.jpg', '*.jpeg', '*.png', '*.bmp', '*.webp']) > 0:
        SAMPLE_SOURCE = source
        break

assert SAMPLE_SOURCE is not None, 'No sample images found for prediction.'
print('Predicting on:', SAMPLE_SOURCE)

!yolo task=detect mode=predict model={BEST_PT} source={SAMPLE_SOURCE} imgsz={IMG_SIZE} conf=0.25 save=True

## 12. Export as `plate_best.pt` and Save to Drive

The local Colab copy will be `/content/plate_best.pt`.

The Drive copy will be `/content/drive/MyDrive/plate_yolo_training/plate_best.pt`.

In [ ]:
import shutil

PLATE_BEST = Path('/content/plate_best.pt')
DRIVE_PLATE_BEST = DRIVE_OUT_DIR / 'plate_best.pt'

shutil.copy2(BEST_PT, PLATE_BEST)
shutil.copy2(BEST_PT, DRIVE_PLATE_BEST)

print('Saved local model:', PLATE_BEST)
print('Saved Drive model:', DRIVE_PLATE_BEST)

## 13. Optional Download

Run this cell if you want to download `plate_best.pt` directly from Colab.

In [ ]:
from google.colab import files
files.download(str(PLATE_BEST))

## Later Flask Integration Note

After training and testing, copy `plate_best.pt` into the Flask project root or configure the app to point to its location.

Keep the existing traffic violation model `best.pt` separate. The plate model should be loaded as a second YOLO model for plate region detection, then OCR should run on the cropped plate region.